In [1]:
# Cell 1: Imports and Data Loading for Classification
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, classification_report

# Load dataset
iris = load_iris()
X_cls, y_cls = iris.data, iris.target
feature_names_cls = iris.feature_names
target_names_cls = iris.target_names

# a. Split the data set into training and test sets (80% train, 20% test)
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_cls, y_cls, test_size=0.2, random_state=42
)

print(f"Classification Train shape: {X_train_c.shape}, Test shape: {X_test_c.shape}")

Classification Train shape: (120, 4), Test shape: (30, 4)


In [2]:
# Cell 2: b. Build the decision tree & c. Check model performances
clf_tree = DecisionTreeClassifier(random_state=42)
clf_tree.fit(X_train_c, y_train_c)

# Predictions
y_train_pred_c = clf_tree.predict(X_train_c)
y_test_pred_c = clf_tree.predict(X_test_c)

# Performance evaluation
print(f"Training Accuracy: {accuracy_score(y_train_c, y_train_pred_c)*100:.2f}%")
print(f"Test Accuracy: {accuracy_score(y_test_c, y_test_pred_c)*100:.2f}%")

Training Accuracy: 100.00%
Test Accuracy: 100.00%


In [ ]:
# Cell 3: Find optimal alpha and prune the tree
path = clf_tree.cost_complexity_pruning_path(X_train_c, y_train_c)
ccp_alphas, impurities = path.ccp_alphas, path.impurities

# Train a tree for each alpha to find the best one on test data
clfs = []
for ccp_alpha in ccp_alphas:
    clf = DecisionTreeClassifier(random_state=42, ccp_alpha=ccp_alpha)
    clf.fit(X_train_c, y_train_c)
    clfs.append(clf)

# Evaluate accuracy vs alpha
train_scores = [accuracy_score(y_train_c, clf.predict(X_train_c)) for clf in clfs]
test_scores = [accuracy_score(y_test_c, clf.predict(X_test_c)) for clf in clfs]

# Pick the alpha that gives optimal test performance (skipping the last one which is a trivial tree)
best_alpha_idx = np.argmax(test_scores[:-1])
best_alpha = ccp_alphas[best_alpha_idx]

# Build final pruned tree
pruned_clf = DecisionTreeClassifier(random_state=42, ccp_alpha=best_alpha)
pruned_clf.fit(X_train_c, y_train_c)

print(f"Optimal Alpha chosen: {best_alpha:.4f}")
print(f"Pruned Train Accuracy: {accuracy_score(y_train_c, pruned_clf.predict(X_train_c))*100:.2f}%")
print(f"Pruned Test Accuracy: {accuracy_score(y_test_c, pruned_clf.predict(X_test_c))*100:.2f}%")